In [1]:
import os
import glob
import sys

import pandas as pd
import scipy.stats as ss
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl

In [2]:
from legnet_embedding import LegNetEmbedding
sys.path.append("../../predictor/model/")
import utrdata_cl as utrdata
from pl_regressor import RNARegressor

## Loading data

In [3]:
utr_type = "utr3"

In [4]:
if utr_type == "utr5":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr5-deltas-epoch=9-step=840.ckpt"
elif utr_type == "utr3":
    MODEL_PATH = "../../predictor/regression_multiple/saved_models/model-utr3-deltas-epoch=9-step=1330.ckpt"
else:
    raise ValueError

In [5]:
PATH_FROM = f'../../predictor/regression_multiple/{utr_type.upper()}_zscores_replicateagg.csv'
df = pd.read_csv(PATH_FROM)

In [6]:
df.head(10)

,seq,cell_type,fold,1,2,3,4,mass_center,mass_center_mean,diff,zscore,mass_center_std
0,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c1,train,6.809279,5.843161,5.836944,23.948600,3.105728,2.860766,0.244962,1.165330,0.210208
1,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c13,train,42.009822,34.532234,32.334772,56.966098,2.628650,2.860766,-0.232116,-1.104220,0.210208
2,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c17,train,50.033644,45.049730,54.291393,110.129539,2.865176,2.860766,0.004410,0.020978,0.210208
3,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c2,train,26.001559,44.713552,22.349163,57.504485,2.739573,2.860766,-0.121193,-0.576539,0.210208
4,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c4,train,12.116972,26.338286,33.479228,67.473281,3.121235,2.860766,0.260469,1.239099,0.210208
5,AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTA...,c6,train,35.742516,30.393594,37.176619,55.042633,2.704235,2.860766,-0.156531,-0.744648,0.210208
6,AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTT...,c1,train,11.557855,31.707073,26.831843,82.272090,3.180150,2.600835,0.579316,1.982492,0.292216
7,AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTT...,c13,train,232.330010,211.524779,171.515311,244.514094,2.497990,2.600835,-0.102845,-0.351948,0.292216
8,AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTT...,c17,train,346.139089,472.864114,433.735295,255.722487,2.397121,2.600835,-0.203714,-0.697134,0.292216
9,AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTT...,c2,train,179.976823,198.152511,131.057403,169.914723,2.428375,2.600835,-0.172460,-0.590179,0.292216


In [7]:
ct_classes = df["cell_type"].unique()
num_classes = ct_classes.shape[0]
num_classes

6

In [8]:
batch_size = 1024

In [9]:
num_workers = 32

In [10]:
test_set = utrdata.UTRData(
    df=df,
    predict_cols=[],
    features=("sequence", "positional", "conditions"),
    construct_type=utr_type,
    augment=False,
    augment_test_time=False,
    augment_kws=dict(
        extend_left=0,
        extend_right=0,
        shift_left=0,
        shift_right=0,
        revcomp=False,
    ),
)

In [11]:
# Creating DataLoaders
dl_test = DataLoader(
    test_set,
    batch_size=batch_size,
    num_workers=num_workers,
    shuffle=False,
    drop_last=False
)

In [12]:
ckpt = torch.load(MODEL_PATH)
ckpt["hyper_parameters"]["model_class"] = LegNetEmbedding

loaded_model = RNARegressor(**ckpt["hyper_parameters"])
loaded_model.load_state_dict(ckpt["state_dict"])

In [15]:
progressbar_callback = pl.callbacks.TQDMProgressBar(refresh_rate=0.5)
trainer = pl.Trainer(
    callbacks=[progressbar_callback],
    logger=False,
    accelerator="gpu",
    devices=[1],
    deterministic=True,
    # gradient_clip_val=1e-5,
    # gradient_clip_algorithm="norm",
)

prediction = trainer.predict(model=loaded_model, dataloaders=dl_test)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA GeForce RTX 3090') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Predicting: 0it [00:00, ?it/s]

In [16]:
pred_tuples, _ = zip(*prediction)
embed, pred = zip(*pred_tuples)
embed = torch.concat(embed).numpy()
pred = torch.concat(pred).numpy()

In [17]:
seq_embedding = embed.reshape(-1, num_classes * embed.shape[-1], order="C")
seq_embedding

array([[ 0.1381321 ,  0.18610378,  0.04100413, ...,  0.18069936,
         0.14377159,  0.24553418],
       [ 0.08550569,  0.1618972 ,  0.11670057, ...,  0.01825945,
         0.07619935,  0.10255897],
       [ 0.12945132,  0.09839492,  0.09263062, ...,  0.2473026 ,
         0.09914529,  0.1593088 ],
       ...,
       [ 0.05874393,  0.14052789,  0.09657188, ...,  0.12070817,
         0.13581255,  0.14626586],
       [ 0.06061119,  0.1236843 ,  0.0665808 , ...,  0.16838956,
         0.09741832,  0.17489882],
       [-0.03064251,  0.13822657,  0.18733846, ...,  0.01650121,
         0.08764598,  0.06394456]], dtype=float32)

In [18]:
pred_mass_center = pred[:, 1].reshape(-1, num_classes)
pred_mass_center

array([[2.8661551, 2.6673675, 2.9867246, 2.859356 , 2.8345942, 2.8976226],
       [2.7966218, 2.5105934, 2.5487952, 2.5169992, 2.5909586, 2.4801164],
       [2.4789352, 2.490981 , 2.6913912, 2.443636 , 2.5885234, 2.6198149],
       ...,
       [2.6879776, 2.5941741, 2.6454806, 2.6211348, 2.6295145, 2.6160395],
       [2.473049 , 2.5459569, 2.5207708, 2.5395153, 2.4919775, 2.5911498],
       [2.6952598, 2.6111164, 2.7318804, 2.6637008, 2.6827042, 2.6018794]],
      dtype=float32)

In [19]:
result = df.pivot(columns="cell_type", index="seq", values="mass_center")
result.columns = pd.MultiIndex.from_product([["mass_center"], result.columns])
result.head()

mass_center            \
cell_type                                                   c1       c13   
seq                                                                        
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTAG...    3.105728  2.628650   
AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTTT...    3.180150  2.497990   
AAAAAAAAAAAAAAAAAAATCCAGCCTGGGCAACATAGTGAATGAAA...    2.583806  2.524408   
AAAAAAAAAAAAAAAAATTCTTAAGTGGGTCTTCATTATGGAAATCA...    2.529623  2.639902   
AAAAAAAAAAAAAAAACTGGTCTTATACAGTTAATTGTTCATCTTAC...    2.976072  2.572164   

                                                                        \
cell_type                                                c17        c2   
seq                                                                      
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTAG...  2.865176  2.739573   
AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTTT...  2.397121  2.428375   
AAAAAAAAAAAAAAAAAAATCCAGCCTGGGCAACATAGTGAATGAAA...  2.599879  2.180035   
AAAAAAAAAAAAAAAAATTCTTAAGTGGGTCTTCATTATGGAAATCA...  2.773180  2.518260   
AAAAAAAAAAAAAAAACTGGTCTTATACAGTTAATTGTTCATCTTAC...  2.722770  2.801453   

                                                                        
cell_type                                                 c4        c6  
seq                                                                     
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTAG...  3.121235  2.704235  
AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTTT...  2.598096  2.503275  
AAAAAAAAAAAAAAAAAAATCCAGCCTGGGCAACATAGTGAATGAAA...  2.753817  2.420384  
AAAAAAAAAAAAAAAAATTCTTAAGTGGGTCTTCATTATGGAAATCA...  2.572179  2.785796  
AAAAAAAAAAAAAAAACTGGTCTTATACAGTTAATTGTTCATCTTAC...  2.648151  2.771069

In [20]:
result[
    pd.MultiIndex.from_product([["pred_mass_center"], result["mass_center"].columns])
] = pred_mass_center
result["pred_mass_center"].head()

cell_type,c1,c13,c17,c2,c4,c6
seq,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTAGATTTAACCAGAATCCCGCCTTCCAGGCTCCTGTTCCCTTTTATTAGCTGCTGAATTTTATTCCCTTTTCCCAACTCCAGGTCCCTCAGTTGTCTTATATCTAAATCCTTTGCCAGGTCCCCACAATCCCAGCACCTGCCTCTGTGAGCTGCCCCCAGGCCCTCACCTTGGCCCTGGCCAGCCTCCTGTGCTCC,2.866155,2.667367,2.986725,2.859356,2.834594,2.897623
AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTTTTAAAACCTGCCTCTGGGTTTTTGTTTTTTCTTGTTTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTCTGAAACAGATCTTGATAAAGCTCTGTGTTGGAGCTGCTGGTTTTTGTTATGGTTGTTGGAATTTCTTGGCCTACTAGGACAGTTCTGTGCTTCACCATGAGGTTTG,2.796622,2.510593,2.548795,2.516999,2.590959,2.480116
AAAAAAAAAAAAAAAAAAATCCAGCCTGGGCAACATAGTGAATGAAACCCCATCTCTACAAAAAATACAAAAATTAGCTGGGCATGATGGTGTGTGCCTGTAGTCCCAGCTACTTGGGGGCTGAGGCAGGAGGATCGCTTGAGCCCGGGAGATCAAGGCTACAGTGAGCTGTGTTCATGCCACTATACCCCAGCCTGGGCAACAAAGTGAGTCCCTGACTCAAAAAATAAAAACAACAAC,2.478935,2.490981,2.691391,2.443636,2.588523,2.619815
AAAAAAAAAAAAAAAAATTCTTAAGTGGGTCTTCATTATGGAAATCACAGAATAGAATGGACAGAATTCTCATTTCTAAAATTCTGTGTGACAGGTTTCATATGGGTCTTTTTATGGTCAGTTACTAAAAGCCACATAATAGTAGAACTAAATTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTGAGACGGAGTCTCGCTCTGTCGCCCAGGCTGGAGTGCAGTGGTGCGATCTTG,2.849767,2.677498,2.906493,2.790464,2.732857,2.881480
AAAAAAAAAAAAAAAACTGGTCTTATACAGTTAATTGTTCATCTTACTATTGGCCAAAGCTAAAATACCTGTGAGCATGTTTCTGTTATTTCTCCTTTCAGATCTTGATTATTTGCCCAGTTGTGTAACATGCCTCAATGTTTTTTAATTGACTGGTGGGCATTGTATTTTAAAAACTATAAAGACAATATATGCAGTAATGAAGGTTTTGTTCTAGCAAGTAGATGGAGTAGAGGCAGA,2.763689,2.576757,2.888324,2.705639,2.715274,2.851203


In [21]:
seq_embedding_df = pd.DataFrame(
    seq_embedding,
    columns=pd.MultiIndex.from_product([
        ["embedding"],
        [f"{ct}_{i:03d}" for ct in result["mass_center"].columns for i in range(embed.shape[-1])]
    ]),
    index=result.index)
result = pd.concat([result, seq_embedding_df], axis=1)

In [22]:
result["embedding"]

,c1_000,c1_001,c1_002,c1_003,c1_004,c1_005,c1_006,c1_007,c1_008,c1_009,...,c6_022,c6_023,c6_024,c6_025,c6_026,c6_027,c6_028,c6_029,c6_030,c6_031
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTAGATTTAACCAGAATCCCGCCTTCCAGGCTCCTGTTCCCTTTTATTAGCTGCTGAATTTTATTCCCTTTTCCCAACTCCAGGTCCCTCAGTTGTCTTATATCTAAATCCTTTGCCAGGTCCCCACAATCCCAGCACCTGCCTCTGTGAGCTGCCCCCAGGCCCTCACCTTGGCCCTGGCCAGCCTCCTGTGCTCC,0.138132,0.186104,0.041004,0.103286,0.035674,0.086632,0.107138,0.172799,0.146467,0.104786,...,-0.008898,0.031258,0.008749,0.142214,0.194688,0.192530,0.118593,0.180699,0.143772,0.245534
AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTTTTAAAACCTGCCTCTGGGTTTTTGTTTTTTCTTGTTTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTCTGAAACAGATCTTGATAAAGCTCTGTGTTGGAGCTGCTGGTTTTTGTTATGGTTGTTGGAATTTCTTGGCCTACTAGGACAGTTCTGTGCTTCACCATGAGGTTTG,0.085506,0.161897,0.116701,0.166041,0.098897,-0.023899,0.229905,0.387182,0.003340,0.004480,...,0.174271,0.192638,0.197231,0.139978,0.059862,0.029033,0.099111,0.018259,0.076199,0.102559
AAAAAAAAAAAAAAAAAAATCCAGCCTGGGCAACATAGTGAATGAAACCCCATCTCTACAAAAAATACAAAAATTAGCTGGGCATGATGGTGTGTGCCTGTAGTCCCAGCTACTTGGGGGCTGAGGCAGGAGGATCGCTTGAGCCCGGGAGATCAAGGCTACAGTGAGCTGTGTTCATGCCACTATACCCCAGCCTGGGCAACAAAGTGAGTCCCTGACTCAAAAAATAAAAACAACAAC,0.129451,0.098395,0.092631,0.051458,0.115194,0.124259,0.028642,0.067143,0.214290,0.125176,...,0.094236,0.106118,0.101262,0.120200,0.103526,0.141900,-0.000392,0.247303,0.099145,0.159309
AAAAAAAAAAAAAAAAATTCTTAAGTGGGTCTTCATTATGGAAATCACAGAATAGAATGGACAGAATTCTCATTTCTAAAATTCTGTGTGACAGGTTTCATATGGGTCTTTTTATGGTCAGTTACTAAAAGCCACATAATAGTAGAACTAAATTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTGAGACGGAGTCTCGCTCTGTCGCCCAGGCTGGAGTGCAGTGGTGCGATCTTG,0.148520,0.157790,0.119491,0.199150,0.091446,0.079888,0.019441,0.229283,0.087301,0.148948,...,0.082691,0.083798,0.078211,0.144074,0.168804,0.150559,0.109203,0.189321,0.077065,0.228111
AAAAAAAAAAAAAAAACTGGTCTTATACAGTTAATTGTTCATCTTACTATTGGCCAAAGCTAAAATACCTGTGAGCATGTTTCTGTTATTTCTCCTTTCAGATCTTGATTATTTGCCCAGTTGTGTAACATGCCTCAATGTTTTTTAATTGACTGGTGGGCATTGTATTTTAAAAACTATAAAGACAATATATGCAGTAATGAAGGTTTTGTTCTAGCAAGTAGATGGAGTAGAGGCAGA,0.147196,0.146589,0.067820,0.101263,0.076544,0.059571,0.189401,0.173141,0.098762,0.077022,...,0.147447,0.063612,0.074593,0.128649,0.162533,0.172967,0.112092,0.216929,0.158011,0.170034
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTTTTTTTTTTTTTTTGAGACGGAGTCTCACTCTGTCGCCAGGCTGGCGCGATCTTGGCTCACTGCAACCTCTGCCTCACGGGTTCAAGCGATTTTCCTGCTTCAGCCTCCTGAGTAGCTGGGACTACAGGCACACGCCACCATGCCCAGCTCATTTTTGTATTTTTAGTAGAGATGGGGTTTCACCATGTTGGCCAGGATGGCCTCCATCTCTTGACCTTGTGATCCGCCCGACTC,0.024637,0.142113,0.093118,0.134696,0.113455,0.081327,0.025997,0.238658,0.078346,0.103178,...,0.085352,0.100374,0.080230,0.130052,0.137506,0.129797,0.116267,0.106408,0.140385,0.164718
TTTTTTTTTTTTTTTTTTTTAAAGACGGGGACTCGCTGTGTTTCCCAGGCTGGAGTGCAGTGCTGCAATCTTGGCTCACTGCAACCTCCATCTCCTAGGTTCAAGCGATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGACGACAGGCACATGCCACCATGCCCAGCTAATTTTTGTATTTTTAGTAGATACGGGGTTTTACCATGTCGGCCAGATGGTCTCAATCTCCTGAACTCATGA,0.056192,0.137441,0.097464,0.126896,0.112046,0.065974,0.064016,0.234009,0.075066,0.093851,...,0.100344,0.093853,0.072740,0.130795,0.128331,0.123158,0.112202,0.120467,0.129330,0.163050
TTTTTTTTTTTTTTTTTTTTTGAGACAGAGTAGCTCTCTGCCAGCCAGGCTGGAGTGCAGTGGCATGATCTCAGCTCGCTGCAACCCCCACCTCCCAGGTTCAAGCGATTCTCCTGTGTCAGTCTCCTGAGTAGCTGGGATTACAGGCACCCACCACTATGCCCAGCTAATTTTTTTATTTTTAGTAGAGACGGTGTTTCACCATGTTGGTCAGGCTGGTCTCGAACTCCTGACCTCATG,0.058744,0.140528,0.096572,0.104248,0.118993,0.061020,0.101497,0.235238,0.074325,0.081819,...,0.127277,0.124940,0.082591,0.133762,0.120880,0.115135,0.098083,0.120708,0.135813,0.146266


### Counting k-mers

In [23]:
sys.path.append("../../benchmark/kmers")
from kmer_model import KmerLinearRegressor

In [24]:
kmer_dfs = []
for k in [1, 2, 3]:
    kmerreg = KmerLinearRegressor(kmer_length=k)
    kmer_df_k = kmerreg.count_kmers(result.index.values)
    kmer_df_k.columns = pd.MultiIndex.from_product([
        ["kmer_counts"],
        kmer_df_k.columns
    ])
    kmer_dfs.append(kmer_df_k)
    print(f"k={k} done")
result = pd.concat([result] + kmer_dfs, axis=1)

k=1 done
k=2 done
k=3 done


In [25]:
result["kmer_counts"]

,A,C,G,T,AA,AC,AG,AT,CA,CC,...,TCG,TCT,TGA,TGC,TGG,TGT,TTA,TTC,TTG,TTT
seq,,,,,,,,,,,,,,,,,,,,,
AAAAAAAAAAAAAAAAAAAAAAAAAAAAAATTTCCATGGCTAACTAGATTTAACCAGAATCCCGCCTTCCAGGCTCCTGTTCCCTTTTATTAGCTGCTGAATTTTATTCCCTTTTCCCAACTCCAGGTCCCTCAGTTGTCTTATATCTAAATCCTTTGCCAGGTCCCCACAATCCCAGCACCTGCCTCTGTGAGCTGCCCCCAGGCCCTCACCTTGGCCCTGGCCAGCCTCCTGTGCTCC,65,78,33,64,37,6,11,11,14,38,...,0,3,2,5,3,4,5,5,3,9
AAAAAAAAAAAAAAAAAAAGGCTTTTCTTTTTAAACAGTTCCACTTTTAAAACCTGCCTCTGGGTTTTTGTTTTTTCTTGTTTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTGTCTGAAACAGATCTTGATAAAGCTCTGTGTTGGAGCTGCTGGTTTTTGTTATGGTTGTTGGAATTTCTTGGCCTACTAGGACAGTTCTGTGCTTCACCATGAGGTTTG,48,29,63,100,28,7,8,5,6,5,...,0,8,3,3,6,32,3,6,10,20
AAAAAAAAAAAAAAAAAAATCCAGCCTGGGCAACATAGTGAATGAAACCCCATCTCTACAAAAAATACAAAAATTAGCTGGGCATGATGGTGTGTGCCTGTAGTCCCAGCTACTTGGGGGCTGAGGCAGGAGGATCGCTTGAGCCCGGGAGATCAAGGCTACAGTGAGCTGTGTTCATGCCACTATACCCCAGCCTGGGCAACAAAGTGAGTCCCTGACTCAAAAAATAAAAACAACAAC,88,54,55,43,46,13,16,13,19,17,...,1,2,8,2,5,5,1,1,2,0
AAAAAAAAAAAAAAAAATTCTTAAGTGGGTCTTCATTATGGAAATCACAGAATAGAATGGACAGAATTCTCATTTCTAAAATTCTGTGTGACAGGTTTCATATGGGTCTTTTTATGGTCAGTTACTAAAAGCCACATAATAGTAGAACTAAATTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTGAGACGGAGTCTCGCTCTGTCGCCCAGGCTGGAGTGCAGTGGTGCGATCTTG,69,32,45,94,32,7,14,16,12,3,...,2,9,2,2,7,3,4,6,2,39
AAAAAAAAAAAAAAAACTGGTCTTATACAGTTAATTGTTCATCTTACTATTGGCCAAAGCTAAAATACCTGTGAGCATGTTTCTGTTATTTCTCCTTTCAGATCTTGATTATTTGCCCAGTTGTGTAACATGCCTCAATGTTTTTTAATTGACTGGTGGGCATTGTATTTTAAAAACTATAAAGACAATATATGCAGTAATGAAGGTTTTGTTCTAGCAAGTAGATGGAGTAGAGGCAGA,79,33,44,84,34,8,15,21,13,6,...,0,6,4,3,5,9,7,5,8,12
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
TTTTTTTTTTTTTTTTTTTGAGACGGAGTCTCACTCTGTCGCCAGGCTGGCGCGATCTTGGCTCACTGCAACCTCTGCCTCACGGGTTCAAGCGATTTTCCTGCTTCAGCCTCCTGAGTAGCTGGGACTACAGGCACACGCCACCATGCCCAGCTCATTTTTGTATTTTTAGTAGAGATGGGGTTTCACCATGTTGGCCAGGATGGCCTCCATCTCTTGACCTTGTGATCCGCCCGACTC,38,68,55,79,2,13,13,10,18,18,...,1,6,4,4,6,4,1,4,6,26
TTTTTTTTTTTTTTTTTTTTAAAGACGGGGACTCGCTGTGTTTCCCAGGCTGGAGTGCAGTGCTGCAATCTTGGCTCACTGCAACCTCCATCTCCTAGGTTCAAGCGATTCTCCTGCCTCAGCCTCCTGAGTAGCTGGGACGACAGGCACATGCCACCATGCCCAGCTAATTTTTGTATTTTTAGTAGATACGGGGTTTTACCATGTCGGCCAGATGGTCTCAATCTCCTGAACTCATGA,46,61,53,80,8,11,14,12,18,16,...,2,5,3,7,4,4,3,3,2,27
TTTTTTTTTTTTTTTTTTTTTGAGACAGAGTAGCTCTCTGCCAGCCAGGCTGGAGTGCAGTGGCATGATCTCAGCTCGCTGCAACCCCCACCTCCCAGGTTCAAGCGATTCTCCTGTGTCAGTCTCCTGAGTAGCTGGGATTACAGGCACCCACCACTATGCCCAGCTAATTTTTTTATTTTTAGTAGAGACGGTGTTTCACCATGTTGGTCAGGCTGGTCTCGAACTCCTGACCTCATG,44,63,53,80,4,11,20,9,20,19,...,2,6,4,4,5,4,3,3,2,28


In [26]:
result.to_parquet(f"embeddings_mapperless_{utr_type}.parquet")